In [ ]:
# IMPORTATION ET NETTOYAGE DES DONNÉES

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Chargement du dataset
df = pd.read_csv("global_power_plant_database.csv", low_memory=False)

print("Dimensions du dataset :", df.shape)
print(df.head())

# Valeurs manquantes
print("\nValeurs manquantes :")
print(df.isnull().sum())

# Colonnes numériques importantes
numeric_cols = [
    "capacity_mw",
    "latitude",
    "longitude",
    "commissioning_year"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remplacement des valeurs manquantes
df["capacity_mw"] = df["capacity_mw"].fillna(df["capacity_mw"].median())
df["commissioning_year"] = df["commissioning_year"].fillna(
    df["commissioning_year"].median()
)

print("\nVérification après nettoyage :")
print(df[numeric_cols].isnull().sum())

Dimensions du dataset : (34936, 36)
  country country_long                                              name  \
0     AFG  Afghanistan      Kajaki Hydroelectric Power Plant Afghanistan   
1     AFG  Afghanistan                                      Kandahar DOG   
2     AFG  Afghanistan                                      Kandahar JOL   
3     AFG  Afghanistan     Mahipar Hydroelectric Power Plant Afghanistan   
4     AFG  Afghanistan  Naghlu Dam Hydroelectric Power Plant Afghanistan   

      gppd_idnr  capacity_mw  latitude  longitude primary_fuel other_fuel1  \
0  GEODB0040538         33.0    32.322    65.1190        Hydro         NaN   
1    WKS0070144         10.0    31.670    65.7950        Solar         NaN   
2    WKS0071196         10.0    31.623    65.7920        Solar         NaN   
3  GEODB0040541         66.0    34.556    69.4787        Hydro         NaN   
4  GEODB0040534        100.0    34.641    69.7170        Hydro         NaN   

  other_fuel2  ... estimated_generatio

In [ ]:
# ANALYSE EXPLORATOIRE DES DONNÉES

print("\nStatistiques descriptives")
print(df.describe())

print("\nMédiane")
print(df.median(numeric_only=True))

# Répartition par pays
country_distribution = df["country_long"].value_counts()

print("\nTop 10 pays")
print(country_distribution.head(10))

# Répartition par type de combustible
fuel_distribution = df["primary_fuel"].value_counts()

print("\nRépartition des combustibles")
print(fuel_distribution)

In [ ]:
# ANALYSE STATISTIQUE

fuel_stats = df.groupby("primary_fuel")["capacity_mw"].agg(
    ["count", "mean", "median", "std"]
)

print(fuel_stats)

# Comparaison simple entre combustibles

fuel_means = df.groupby("primary_fuel")["capacity_mw"].mean()

print("\nCapacité moyenne par combustible")
print(fuel_means.sort_values(ascending=False))

In [ ]:
# TEST D'HYPOTHÈSE

# Hypothèse :
# H0 : les centrales à charbon et hydroélectriques
# ont la même capacité moyenne

coal = df[df["primary_fuel"] == "Coal"]["capacity_mw"].dropna()
hydro = df[df["primary_fuel"] == "Hydro"]["capacity_mw"].dropna()

coal_mean = np.mean(coal)
hydro_mean = np.mean(hydro)

print("Moyenne Coal :", coal_mean)
print("Moyenne Hydro :", hydro_mean)

difference = abs(coal_mean - hydro_mean)

print("Différence :", difference)

if difference > 50:
    print("On observe une différence importante entre les deux groupes.")
else:
    print("La différence observée reste faible.")

In [ ]:
# ANALYSE DES SÉRIES TEMPORELLES

plants_per_year = (
    df.groupby("commissioning_year")
      .size()
      .sort_index()
)

print(plants_per_year.tail())

plt.figure(figsize=(12,6))
plants_per_year.plot()

plt.title("Nombre de centrales mises en service par année")
plt.xlabel("Année")
plt.ylabel("Nombre de centrales")
plt.grid(True)

plt.show()

In [ ]:
# ÉVOLUTION DES COMBUSTIBLES

fuel_year = pd.crosstab(
    df["commissioning_year"],
    df["primary_fuel"]
)

fuel_year.tail()

fuel_year.plot(
    figsize=(14,8)
)

plt.title("Évolution des types de combustibles")
plt.xlabel("Année")
plt.ylabel("Nombre de centrales")

plt.show()

In [ ]:
# VISUALISATIONS AVANCÉES

# Top combustibles

plt.figure(figsize=(10,6))

sns.countplot(
    y="primary_fuel",
    data=df,
    order=df["primary_fuel"].value_counts().index
)

plt.title("Répartition des centrales par combustible")

plt.show()

In [ ]:
# Carte géographique simplifiée

plt.figure(figsize=(12,8))

plt.scatter(
    df["longitude"],
    df["latitude"],
    alpha=0.3,
    s=10
)

plt.title("Répartition géographique des centrales")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

In [ ]:
# OPÉRATIONS MATRICIELLES

matrix_data = df[
    ["capacity_mw", "latitude", "longitude"]
].dropna()

matrix = matrix_data.to_numpy()

# Matrice de covariance

cov_matrix = np.cov(matrix.T)

print("Matrice de covariance")
print(cov_matrix)

# Valeurs propres et vecteurs propres

eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

print("\nValeurs propres")
print(eigenvalues)

print("\nVecteurs propres")
print(eigenvectors)

In [ ]:
# INTÉGRATION NUMPY ET PANDAS


capacity = df["capacity_mw"].to_numpy()

high_capacity = capacity > np.percentile(capacity, 90)

top_plants = df[high_capacity]

print(top_plants[
    ["name", "country_long", "capacity_mw"]
].head())

In [ ]:
# NUMPY + MATPLOTLIB

capacity = df["capacity_mw"].dropna().to_numpy()

plt.figure(figsize=(10,6))

plt.hist(
    capacity,
    bins=50
)

plt.title("Distribution des capacités")
plt.xlabel("Capacité (MW)")
plt.ylabel("Nombre de centrales")

plt.show()